<a href="https://colab.research.google.com/github/zumrakcck/Fake-News-Detection-with-Deep-Learning/blob/main/FakeNewDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Veri Ön İşleme (Preprocessing) ve BiLSTM Taban Modeli (Baseline)


In [ ]:
# 1. Install kagglehub to download the ISOT dataset automatically
!pip install -q kagglehub

import kagglehub
import shutil
import os
import pandas as pd
import re
from sklearn.model_selection import train_test_split

print("Downloading the dataset...")
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
print("Download complete. Files are located at:", path)

# Copy CSV files to the current working directory for Pandas
for file in os.listdir(path):
    if file.endswith('.csv'):
        shutil.copy(os.path.join(path, file), '.')
        print(f"{file} copied to working directory!")

# 2. Load the datasets[cite: 17]
df_fake = pd.read_csv('Fake.csv')
df_true = pd.read_csv('True.csv')

# Assign binary labels: Fake = 1, True = 0[cite: 17]
df_fake['label'] = 1
df_true['label'] = 0

# Concatenate and shuffle the dataset
df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total raw data count: {len(df)}")

# 3. Combine title and text (truncated/merged as specified in the proposal)[cite: 17]
df['content'] = df['title'].fillna('') + " " + df['text'].fillna('')

# Remove duplicate texts before splitting[cite: 17]
df = df.drop_duplicates(subset=['content']).reset_index(drop=True)
print(f"Data count after removing duplicates: {len(df)}")

# 4. Source-cue check implementation[cite: 17]
# Removes obvious source cues (URLs, Reuters, news agency names) for the artifact-aware experiment
def remove_source_cues(text):
    text = re.sub(r'^[A-Z\s,]+\s?\(Reuters\)\s?-?', '', text, flags=re.IGNORECASE)
    text = re.sub(r'reuters', '', text, flags=re.IGNORECASE)
    text = re.sub(r'associated press', '', text, flags=re.IGNORECASE)
    return text

df['cleaned_content'] = df['content'].apply(remove_source_cues)

# 5. Stratified Train / Validation / Test Splits (70% / 15% / 15%)[cite: 17]
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['content'].tolist(), df['label'].tolist(), test_size=0.3, random_state=42, stratify=df['label']
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

print(f"--- Dataset Splits Ready ---")
print(f"Train size: {len(train_texts)}")
print(f"Validation size: {len(val_texts)}")
print(f"Test size: {len(test_texts)}")

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Download complete. Files are located at: /kaggle/input/fake-and-real-news-dataset
True.csv copied to working directory!
Fake.csv copied to working directory!
Total raw data count: 44898
Data count after removing duplicates: 39105
--- Dataset Splits Ready ---
Train size: 27373
Validation size: 5866
Test size: 5866


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# 1. Hiperparametreler
MAX_VOCAB_SIZE = 20000
MAX_LEN = 300  # Metinlerin kırpılacağı/uzatılacağı maksimum kelime sayısı
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 0.001

# Cihaz ayarı (GPU varsa GPU'yu kullanır)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Kullanılan cihaz: {device}")

# 2. Basit Kelime Sözlüğü (Vocabulary) Oluşturma
tokenizer = lambda x: x.lower().split()

counter = Counter()
for text in train_texts:
    counter.update(tokenizer(text))

# En sık geçen kelimeleri seçelim
vocab = {word: i + 2 for i, (word, _) in enumerate(counter.most_common(MAX_VOCAB_SIZE))}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

def text_to_sequence(text, vocab, max_len):
    tokens = tokenizer(text)
    seq = [vocab.get(token, vocab['<unk>']) for token in tokens]
    if len(seq) < max_len:
        seq = seq + [vocab['<pad>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

# Eğitim, doğrulama ve test verilerini dönüştürme
X_train = [text_to_sequence(t, vocab, MAX_LEN) for t in train_texts]
X_val = [text_to_sequence(t, vocab, MAX_LEN) for t in val_texts]
X_test = [text_to_sequence(t, vocab, MAX_LEN) for t in test_texts]

# 3. PyTorch Dataset Sınıfı
class NewsDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# train_labels değişkenini doğrudan bağlıyoruz (hatalı olan ilk satır kaldırıldı):
train_loader = DataLoader(NewsDataset(X_train, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(NewsDataset(X_val, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(NewsDataset(X_test, test_labels), batch_size=BATCH_SIZE, shuffle=False)

# 4. BiLSTM Model Mimarisi
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Çift yönlü (bidirectional) olduğu için ileri ve geri hidden state'leri birleştiriyoruz
        hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.fc(hidden_cat)
        return self.sigmoid(out)

model = BiLSTMClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("BiLSTM modeli başarıyla tanımlandı!")

Kullanılan cihaz: cuda
BiLSTM modeli başarıyla tanımlandı!


In [ ]:
# 5. BiLSTM Eğitim Döngüsü (Training Loop)
print("Eğitim başlıyor...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for sequences, labels in train_loader:
        sequences, labels = sequences.to(device), labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader):.4f}")

# 6. Modelin Test Edilmesi ve Metriklerin Çıkarılması
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        outputs = model(sequences)
        preds = (outputs >= 0.5).float().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Metrikleri hesaplama
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

print("\n--- BiLSTM Baseline Test Sonuçları ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Macro F1:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

Eğitim başlıyor...
Epoch [1/3], Loss: 0.0750
Epoch [2/3], Loss: 0.0071
Epoch [3/3], Loss: 0.0117

--- BiLSTM Baseline Test Sonuçları ---
Accuracy:  0.9990
Macro F1:  0.9990
Precision: 0.9989
Recall:    0.9990


In [ ]:
# Temizlenmiş içerikler için veri bölme (train/val/test) adımı
train_texts_clean, temp_texts_clean, _, _ = train_test_split(
    df['cleaned_content'].tolist(), df['label'].tolist(), test_size=0.3, random_state=42, stratify=df['label']
)
val_texts_clean, test_texts_clean, _, _ = train_test_split(
    temp_texts_clean, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

# Sekanslara dönüştürme
X_train_clean = [text_to_sequence(t, vocab, MAX_LEN) for t in train_texts_clean]
X_val_clean = [text_to_sequence(t, vocab, MAX_LEN) for t in val_texts_clean]
X_test_clean = [text_to_sequence(t, vocab, MAX_LEN) for t in test_texts_clean]

# DataLoader oluşturma
train_loader_clean = DataLoader(NewsDataset(X_train_clean, train_labels), batch_size=BATCH_SIZE, shuffle=True)
test_loader_clean = DataLoader(NewsDataset(X_test_clean, test_labels), batch_size=BATCH_SIZE, shuffle=False)

# Yeni bir model nesnesi oluşturup temizlenmiş veriyle eğitiyoruz
model_clean = BiLSTMClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM).to(device)
criterion_clean = nn.BCELoss()
optimizer_clean = optim.Adam(model_clean.parameters(), lr=LEARNING_RATE)

print("Temizlenmiş veri ile BiLSTM eğitimi başlıyor...")
for epoch in range(EPOCHS):
    model_clean.train()
    total_loss = 0
    for sequences, labels in train_loader_clean:
        sequences, labels = sequences.to(device), labels.to(device).unsqueeze(1)
        optimizer_clean.zero_grad()
        outputs = model_clean(sequences)
        loss = criterion_clean(outputs, labels)
        loss.backward()
        optimizer_clean.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader_clean):.4f}")

# Temizlenmiş modelin test sonuçları
model_clean.eval()
all_preds_clean = []
all_labels_clean = []

with torch.no_grad():
    for sequences, labels in test_loader_clean:
        sequences = sequences.to(device)
        outputs = model_clean(sequences)
        preds = (outputs >= 0.5).float().cpu().numpy()
        all_preds_clean.extend(preds)
        all_labels_clean.extend(labels.numpy())

accuracy_clean = accuracy_score(all_labels_clean, all_preds_clean)
precision_clean, recall_clean, f1_clean, _ = precision_recall_fscore_support(all_labels_clean, all_preds_clean, average='macro')

print("\n--- Kaynak İpuçları Temizlenmiş BiLSTM Test Sonuçları ---")
print(f"Accuracy:  {accuracy_clean:.4f}")
print(f"Macro F1:  {f1_clean:.4f}")
print(f"Precision: {precision_clean:.4f}")
print(f"Recall:    {recall_clean:.4f}")

Temizlenmiş veri ile BiLSTM eğitimi başlıyor...
Epoch [1/3], Loss: 0.1688
Epoch [2/3], Loss: 0.0416
Epoch [3/3], Loss: 0.0198

--- Kaynak İpuçları Temizlenmiş BiLSTM Test Sonuçları ---
Accuracy:  0.9889
Macro F1:  0.9888
Precision: 0.9886
Recall:    0.9891
